In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
import os

prompt = ChatPromptTemplate.from_template('Tell me a fun fact about {topic}.') 
load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")
if api_key is None: raise ValueError("ANTHROPIC_API_KEY not found in .env file")
base_url=os.getenv("BASE_URL")
model_name=os.getenv("MODEL_NAME")
 
llm = ChatAnthropic(model=model_name,base_url=base_url,anthropic_api_key=api_key, max_tokens=200)

chain = prompt | llm          # prompt --> llm  
 
response = chain.invoke({'topic': 'space'}) 
print(type(response).__name__)    # AIMessage 
print(type(response))
#print(response.content) 
chain = prompt | llm  | StrOutputParser()   # prompt --> llm --> parser 
 
response = chain.invoke({'topic': 'black holes'}) 
print(type(response))    # <class 'str'>  (plain string, not AIMessage) 
#print(response) 


AIMessage
<class 'langchain_core.messages.ai.AIMessage'>
<class 'langchain_core.messages.base.TextAccessor'>


In [8]:
# Step 1: summarise a topic 
summary_prompt = ChatPromptTemplate.from_template('Summarize {topic} in 2 sentences.') 
summary_chain  = summary_prompt | llm | StrOutputParser() 
 
# Step 2: turn that summary into a tweet 
tweet_prompt   = ChatPromptTemplate.from_template('Turn this into a tweet:\n{summary}') 
tweet_chain    = tweet_prompt | llm | StrOutputParser() 
 
# Connect: summary output --> tweet input 
full_chain = summary_chain | (lambda s: {'summary': s}) | tweet_chain 
 
result = full_chain.invoke({'topic': 'Quantum Computing'}) 
print(result) 

Here's the tweet:

"Quantum computers use qubits that exist in multiple states at once (thanks to superposition & entanglement) — solving complex problems EXPONENTIALLY faster than classical computers. 🚀 Think breakthroughs in cryptography, drug discovery & optimization. The future is quantum. ⚛️ #QuantumComputing #Tech"


In [ ]:
#Parallel Chain 
from langchain_core.runnables import RunnableParallel 
 
pros_prompt = ChatPromptTemplate.from_template('List 2 pros of {technology}.') 
cons_prompt = ChatPromptTemplate.from_template('List 2 cons of {technology}.') 
 
# Both chains run IN PARALLEL 
parallel_chain = RunnableParallel( 
    pros = pros_prompt | llm | StrOutputParser(), 
    cons = cons_prompt | llm | StrOutputParser(), 
)  
result = parallel_chain.invoke({'technology': 'AI'}) 

print('PROS:\n', result['pros']) 
print('CONS:\n', result['cons']) 



PROS:
 Here are 2 pros of AI:

1. **Increased Efficiency** - AI can automate repetitive tasks and process large amounts of data much faster than humans, saving time and reducing errors in fields like healthcare, finance, and manufacturing.

2. **Enhanced Decision Making** - AI can analyze vast amounts of information to identify patterns and insights that humans might miss, helping businesses and organizations make better-informed decisions.

CONS:
 Here are 2 cons of AI:

1. **Job Displacement** - AI automation can replace human workers in various industries, potentially leading to unemployment and economic disruption for those whose skills become less in demand.

2. **Privacy and Security Risks** - AI systems often rely on large amounts of personal data, raising concerns about data privacy, surveillance, and the potential for misuse or security breaches.


In [ ]:
translate_prompt = ChatPromptTemplate.from_template('Translate to French: {text}') 
summarize_prompt = ChatPromptTemplate.from_template('Summarize in one sentence: {translated}') 
 
pipeline = ( 
    translate_prompt | llm | StrOutputParser() 
    | (lambda t: {'translated': t}) 
    | summarize_prompt | llm | StrOutputParser() 
) 
 
result = pipeline.invoke({'text': 'LangChain helps build LLM-powered applications easily.'}) 
print(result) 

LangChain simplifies the development of LLM-powered applications.


In [9]:
qa_prompt = ChatPromptTemplate.from_messages([ 
    ('system', 'You are an expert {domain} teacher. Give short, clear answers.'), 
    ('human',  '{question}'), 
]) 
 
qa_chain = qa_prompt | llm | StrOutputParser() 
 
# Batch: ask multiple questions in parallel 
questions = [ 
    {'domain': 'Python',       'question': 'What is a decorator?'}, 
    {'domain': 'Data Science', 'question': 'What is overfitting?'}, 
] 
 
answers = qa_chain.batch(questions) 
for q, a in zip(questions, answers): 
    print(f"Q: {q['question']}") 
    print(f"A: {a}") 
    print('-' * 50) 

Q: What is a decorator?
A: ## Python Decorator

A decorator is a function that **wraps another function** to extend or modify its behavior, without changing its source code.

### Basic Syntax

```python
def my_decorator(func):
    def wrapper():
        print("Before")
        func()
        print("After")
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")

say_hello()
# Before
# Hello!
# After
```

The `@my_decorator` syntax is shorthand for:
```python
say_hello = my_decorator(say_hello)
```

---

### Practical Example — Timing a function

```python
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"Took {time
--------------------------------------------------
Q: What is overfitting?
A: ## Overfitting

Overfitting occurs when a model **learns the training data too well** — including its noise and random fluctuations — rather than learning the underlying pattern.

### Res